# CoLA Data Sampling Walkthrough

One notebook that documents both preprocessing (Step 1) and clustering/export (Step 2) so you can sanity-check every stage without rerunning the heavy scripts.

## Pipeline at a Glance

1. **Preprocess evaluable languages** (`python data_prep/preprocess_evaluable_languages.py`)
   - Filter FineWeb-2 to the 108-language evaluable pool (Belebele/FLORES overlap, train split, valid family).
   - Merge `lang_resource_dataset.tsv` → High/Medium/Low buckets to balance sampling tiers.
   - Build one-hot (structural) + LLM (semantic) embeddings, compute deterministic t-SNE projections, cache all artifacts under `processed_artifacts/`.
2. **Cluster + export benchmark subsets** (`python data_prep/create_benchmark_samples.py`)
   - Load cached artifacts, run Silhouette analysis to pick sample sizes, run KMeans, order medoids, and save nested subsets under `onehot_benchmark_samples/` + `llm_benchmark_samples/`.
   - Export three tiers by default: small, medium, and a *full* tier that contains every evaluable language (≈190).
   - Produce CoLA-specific A×B samples (group medoids with nearest neighbors) in `cola_optimal_samples/`.
   - Write interactive t-SNE plots with medoids highlighted next to each CSV.

**Why this split?** Preprocessing is expensive (remote embeddings) but deterministic; caching keeps later experiments fast. Clustering depends on experiment-specific `k`, so it stays in the second step.

### Scientific reasoning (advantages / trade-offs)
- ✅ Reproducibility: cached embeddings/t-SNEs ensure downstream sampling is deterministic until raw data changes.
- ✅ Coverage-aware: filtering to evaluable languages keeps every subset measurable on Belebele/FLORES.
- ✅ Resource context: tier metadata + t-SNE plots surface imbalances early.
- ⚠️ t-SNE distortion: 2D projections preserve local neighborhoods but can warp global structure.
- ⚠️ Cache staleness: rerun preprocessing whenever FineWeb metadata or the embedding endpoint changes.
- ⚠️ Remote dependency: LLM embeddings require the `nomic-embed-text-v2-moe` endpoint.
- Small/medium tiers correspond to the strongest silhouette peaks around k=10 and k≈64; the full tier is always the entire evaluable pool (e.g., 191 languages).


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

try:
    NOTEBOOK_DIR = Path(__file__).parent.resolve()
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

candidate_dirs = [
    NOTEBOOK_DIR / "processed_artifacts",
    NOTEBOOK_DIR / ".." / "processed_artifacts",
    NOTEBOOK_DIR / "data_prep" / "processed_artifacts",
    NOTEBOOK_DIR / ".." / "data_prep" / "processed_artifacts",
]
DATA_DIR = next((p.resolve() for p in candidate_dirs if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "processed_artifacts/ not found. Run preprocess_evaluable_languages.py first or set DATA_DIR manually."
    )

metadata = pd.read_csv(DATA_DIR / "filtered_languages.csv")
onehot_tsne = pd.read_csv(DATA_DIR / "onehot_tsne.csv")
llm_tsne = pd.read_csv(DATA_DIR / "llm_tsne.csv")

print(f"Loaded {len(metadata)} evaluable languages.")
metadata.head()

In [ ]:
print("Resource tier distribution:\n")
display(metadata["resource_category"].value_counts())

print("\nTop scripts:")
display(metadata["script"].value_counts().head(10))

print("\nTop families:")
display(metadata["family"].value_counts().head(10))

### Interactive t-SNE with medoids (after Step 2)
Use the cell below to render the Plotly HTML files generated after clustering.

## Inspect clustering outputs (Step 2)



In [ ]:
def show_tsne_with_medoids(folder):
    base_dir = Path(NOTEBOOK_DIR)
    htmls = sorted((base_dir / folder).glob('*_tsne.html'))
    if not htmls:
        print(f'No t-SNE HTML files found in {folder}. Run create_benchmark_samples.py first.')
        return
    import IPython
    for html_path in htmls:
        print(f'Displaying {html_path.name}')
        display(IPython.display.HTML(filename=str(html_path)))

show_tsne_with_medoids('onehot_benchmark_samples')
show_tsne_with_medoids('llm_benchmark_samples')


In [ ]:
def list_samples(folder):
    path = Path(folder)
    if not path.exists():
        print(f"{folder} missing (run create_benchmark_samples.py).")
        return []
    csvs = sorted(path.glob("*.csv"))
    for csv_path in csvs:
        print(f"- {csv_path.name}")
    return csvs

print("One-hot samples:")
onehot_files = list_samples(Path(NOTEBOOK_DIR) / "onehot_benchmark_samples")
print("\nLLM samples:")
llm_files = list_samples(Path(NOTEBOOK_DIR) / "llm_benchmark_samples")
print("\nCoLA-optimal samples:")
cola_files = list_samples(Path(NOTEBOOK_DIR) / "cola_optimal_samples")

In [ ]:
def preview_csv(path_list, title):
    if not path_list:
        print(f"No files to preview for {title}.")
        return
    df = pd.read_csv(path_list[0])
    print(f"Previewing {path_list[0].name} ({len(df)} rows)")
    display(df.head())

preview_csv(onehot_files, "one-hot")
preview_csv(llm_files, "llm")
preview_csv(cola_files, "CoLA")

## Interpreting the samples

- **Nested medoids** (`*_representative_mediods.csv`): each file contains the first *k* medoids ordered via farthest-first traversal, so larger tiers strictly superset smaller ones.
- The interactive Plotly HTMLs generated during Step 2 (e.g., `llm_k40_tsne.html`) include medoid markers and hover tooltips mirroring `fineweb2_medoid_clustering.py`; open them in a browser for spatial intuition.- The largest tier in each embedding family always equals the total number of evaluable languages, ensuring you can grab the full ~190-lang list without extra bookkeeping.
